# DATA 266 - Homework 6
## Self-Supervised and Contrastive Representation Learning on STL-10

In this notebook I compare three ways of training the same ResNet-18 encoder on STL-10 when only a few labels are available:

1. **Supervised:** train ResNet-18 end to end on just 500 labeled images (10% of the labeled train split).
2. **Rotation SSL:** pretrain on all 100,000 unlabeled images by predicting how much each image was rotated, then freeze the encoder and train only a linear classifier on the same 500 labeled images.
3. **SimCLR:** pretrain on 20,000 unlabeled images with a contrastive loss on two augmented views of each image, then freeze the encoder and train only a linear classifier on the same 500 labeled images.

At the end I look at nearest neighbors in embedding space for the same query images under all three encoders.

I used Demo_6_Self_Supervised_Learning.ipynb as my starting point. Most of the code (the rotation dataset, the two-view dataset, the NT-Xent loss, the embedding function, and the display helpers) comes from the demo, adapted to this assignment:

| | Demo | This notebook |
| --- | --- | --- |
| Dataset | CIFAR-10 (32×32) | STL-10 (96×96), labeled train, test, and unlabeled splits |
| Encoder | TinyEncoder (3 conv layers, 128-d) | ResNet-18 from torchvision, no pretrained weights, 512-d output |
| Methods | rotation SSL and SimCLR | supervised baseline, rotation SSL, and SimCLR |
| Evaluation | nearest neighbors only | linear evaluation with a frozen encoder, test accuracy, and nearest neighbors |
| SimCLR augmentations | RandomResizedCrop, HorizontalFlip, ColorJitter, Grayscale at 32 px | the same four augmentations with the same settings, at 96 px |
| NT-Xent temperature | 0.2 | 0.2 |

## Personal Parameters

These are the same personal parameters I have used all semester. SEED controls everything random in this notebook: Python, NumPy, and PyTorch, the choice of the 500 labeled images, the choice of the 20,000 SimCLR images, data loader shuffling, and the random rotations and augmentations. SLICE, CLS_A, and CLS_B are used to pick the query images for the nearest-neighbor part (CLS_A = 6 is horse and CLS_B = 0 is airplane in STL-10). HP_ID is only reported here.

In [ ]:
SID4 = 9486
SEED = 9486
SLICE = 486
HP_ID = 0
CLS_A = 6
CLS_B = 0

print(f"SID4  = {SID4}")
print(f"SEED  = {SEED}")
print(f"SLICE = {SLICE}")
print(f"HP_ID = {HP_ID}")
print(f"CLS_A = {CLS_A}")
print(f"CLS_B = {CLS_B}")

## Setup: Imports, Device, Reproducibility, and Logging

Same imports as the demo, plus a few for saving results. set_seed seeds Python, NumPy, and PyTorch (CPU and CUDA), and I also turn on cuDNN's deterministic mode. DataLoader workers get their own seeds through seed_worker, and every loader gets a torch.Generator seeded with SEED, so the shuffling order and the random rotations and augmentations done inside the workers are reproducible too.

The log function prints a line with a UTC timestamp and also appends it to results/run_log_raw.txt, so I have the actual console output from the run for RUN_LOG.txt.

In [ ]:
import hashlib
import json
import os
import random
import time
from datetime import datetime, timezone

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset, Subset

import matplotlib.pyplot as plt

for folder in ["figures", "results", "checkpoints"]:
    os.makedirs(folder, exist_ok=True)

LOG_PATH = "results/run_log_raw.txt"
open(LOG_PATH, "w").close()  # start a fresh log for this run


def log(*args):
    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    line = f"[{stamp}] " + " ".join(str(a) for a in args)
    print(line)
    with open(LOG_PATH, "a") as f:
        f.write(line + "\n")


device = "cuda" if torch.cuda.is_available() else "cpu"
log("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "no CUDA device")
log("torch", torch.__version__, "| torchvision", torchvision.__version__)


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_generator(seed=SEED):
    g = torch.Generator()
    g.manual_seed(seed)
    return g


torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
set_seed(SEED)

### Experiment Settings

All the epochs, batch sizes, and learning rates are in one place so the three methods are easy to compare. The epochs follow what the assignment asks for: 15 for the supervised model, 15 for rotation pretraining, 20 for SimCLR pretraining, and 20 for each linear evaluation. The pretraining learning rate (3e-4 with Adam and weight decay 1e-4) is the same as the demo. The labeled-data parts use a higher learning rate (1e-3) because they only get 16 steps per epoch on 500 images.

In [ ]:
NUM_LABELED = 500             # 10% of the 5,000 labeled training images
SIMCLR_SUBSET_SIZE = 20000    # unlabeled images used for SimCLR

BATCH_LABELED = 32            # Part A training and both linear evaluations
EPOCHS_SUP = 15
LR_SUP = 1e-3

BATCH_ROT = 256
EPOCHS_ROT = 15
LR_ROT = 3e-4                 # same as the demo

BATCH_SIM = 256
EPOCHS_SIM = 20
LR_SIM = 3e-4                 # same as the demo
TAU = 0.2                     # NT-Xent temperature, same as the demo

EPOCHS_LIN = 20
LR_LIN = 1e-3

WEIGHT_DECAY = 1e-4           # same as the demo
NUM_WORKERS = 2
PIN_MEMORY = device == "cuda"

## Loading STL-10

STL-10 has three splits that matter here:

- **train:** 5,000 labeled images (500 per class)
- **test:** 8,000 labeled images (800 per class)
- **unlabeled:** 100,000 images with no labels (torchvision gives them the label -1)

All images are 96×96 color images, so they look a lot less pixelated than CIFAR-10. Like the demo, I keep "raw" versions of the datasets that return PIL images, so I can apply different transforms to them for each method and also display them.

The download is about 2.6 GB, so the first run of this cell takes a few minutes.

In [ ]:
classes = ["airplane", "bird", "car", "cat", "deer", "dog", "horse", "monkey", "ship", "truck"]
data_dir = "./data"

t0 = time.time()
train_raw = torchvision.datasets.STL10(data_dir, split="train", download=True, transform=None)
test_raw = torchvision.datasets.STL10(data_dir, split="test", download=True, transform=None)
unlabeled_raw = torchvision.datasets.STL10(data_dir, split="unlabeled", download=True, transform=None)
log(f"STL-10 loaded in {time.time() - t0:.1f}s")
log("train:", len(train_raw), "| test:", len(test_raw), "| unlabeled:", len(unlabeled_raw))

Quick checks on the image shapes and the label ranges before using anything.

In [ ]:
img0, y0 = train_raw[0]
log("one image:", type(img0).__name__, img0.size, img0.mode, "| raw array shape per split:",
    train_raw.data.shape[1:], test_raw.data.shape[1:], unlabeled_raw.data.shape[1:])
log("train labels:", sorted(set(train_raw.labels.tolist())))
log("test labels:", sorted(set(test_raw.labels.tolist())))
log("unlabeled labels:", sorted(set(unlabeled_raw.labels.tolist())))

assert train_raw.data.shape[1:] == test_raw.data.shape[1:] == unlabeled_raw.data.shape[1:] == (3, 96, 96)
assert set(train_raw.labels.tolist()) == set(range(10)) and set(test_raw.labels.tolist()) == set(range(10))
assert set(unlabeled_raw.labels.tolist()) == {-1}
print("Image shapes and label ranges look correct.")

### Input Transform for ResNet-18

ResNet-18 ends with an adaptive average pooling layer, so it works on any input size and I can keep STL-10 at its native 96×96 instead of upscaling to 224×224. Upscaling would not add any information, and it would make every forward pass about 5 times more expensive, which matters for pretraining on 100,000 images.

For normalization, I compute the per-channel mean and standard deviation from the 5,000 labeled training images, the same way the demo uses CIFAR-10's own statistics. test_tf only converts to a tensor and normalizes, with no augmentation, and is used for test evaluation and for the embeddings in Part D.

In [ ]:
pixels = train_raw.data.astype(np.float32) / 255.0  # [N, 3, 96, 96]
mean = tuple(round(float(v), 4) for v in pixels.mean(axis=(0, 2, 3)))
std = tuple(round(float(v), 4) for v in pixels.std(axis=(0, 2, 3)))
del pixels
log("STL-10 mean:", mean, "| std:", std)

test_tf = T.Compose([
    T.ToTensor(),
    T.Normalize(mean, std),
])

test_tensor = torchvision.datasets.STL10(data_dir, split="test", download=False, transform=test_tf)
test_loader = DataLoader(test_tensor, batch_size=256, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
print("Test tensor shape:", tuple(test_tensor[0][0].shape))

## Visualization Helpers

These are the demo's helpers, unchanged except for the default image size. They show images with nearest-neighbor interpolation, so nothing gets blurred when it is displayed larger.

In [ ]:
def show_pil_row(pil_images, titles=None, figsize=(12, 3)):
    fig, axes = plt.subplots(1, len(pil_images), figsize=figsize)
    if len(pil_images) == 1:
        axes = [axes]
    for i, ax in enumerate(axes):
        ax.imshow(pil_images[i], interpolation="nearest")
        ax.axis("off")
        if titles is not None:
            ax.set_title(titles[i])
    plt.show()


def tensor_to_display_img(x):
    # x: normalized tensor [3,H,W]
    x = x.detach().cpu().clone()
    for c in range(3):
        x[c] = x[c] * std[c] + mean[c]
    x = x.clamp(0, 1)
    return x.permute(1, 2, 0).numpy()


def show_tensor_row(tensors, titles=None, figsize=(12, 3)):
    fig, axes = plt.subplots(1, len(tensors), figsize=figsize)
    if len(tensors) == 1:
        axes = [axes]
    for i, ax in enumerate(axes):
        ax.imshow(tensor_to_display_img(tensors[i]), interpolation="nearest")
        ax.axis("off")
        if titles is not None:
            ax.set_title(titles[i])
    plt.show()


imgs = [test_raw[i][0] for i in range(6)]
titles = [classes[test_raw[i][1]] for i in range(6)]
show_pil_row(imgs, titles=titles, figsize=(14, 3))

## The 500-Image Labeled Subset

All three methods only get to see labels for the same 500 training images. I pick them with a NumPy generator seeded with SEED, taking 50 random images from each of the 10 classes. That way the subset is exactly 10% of the labeled split and stays balanced, so no class is left out by chance.

make_labeled_loader is the only way the notebook builds a labeled training loader. It always uses LABELED_IDX, and Parts A, B, and C all call it, so they all train on the same 500 images. The short fingerprint (an MD5 hash of the index list) gets printed each time, which makes this easy to check.

In [ ]:
rng = np.random.default_rng(SEED)
train_labels = np.array(train_raw.labels)
LABELED_IDX = []
for c in range(10):
    class_idx = np.where(train_labels == c)[0]
    LABELED_IDX.extend(rng.choice(class_idx, size=NUM_LABELED // 10, replace=False).tolist())
LABELED_IDX = sorted(int(i) for i in LABELED_IDX)
LABELED_HASH = hashlib.md5(np.array(LABELED_IDX).tobytes()).hexdigest()[:12]

assert len(LABELED_IDX) == NUM_LABELED == len(set(LABELED_IDX)) == 500
per_class = np.bincount(train_labels[LABELED_IDX], minlength=10)
assert (per_class == 50).all()

log(f"Labeled subset: {len(LABELED_IDX)} images, per class {per_class.tolist()}, fingerprint {LABELED_HASH}")
log("First 10 labeled indices:", LABELED_IDX[:10])

For the labeled data I use a light augmentation, a random crop with padding plus a horizontal flip. This is the same idea as the demo's rotation transform, scaled from 32 to 96 pixels. The same transform is used for Part A and for both linear evaluations, so the only difference between the three methods is how the encoder was trained.

In [ ]:
train_tf = T.Compose([
    T.RandomCrop(96, padding=12),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(mean, std),
])

train_aug = torchvision.datasets.STL10(data_dir, split="train", download=False, transform=train_tf)


def make_labeled_loader():
    labeled_ds = Subset(train_aug, LABELED_IDX)
    assert list(labeled_ds.indices) == LABELED_IDX and len(labeled_ds) == 500
    print(f"Using the {len(labeled_ds)}-image labeled subset (fingerprint {LABELED_HASH})")
    return DataLoader(labeled_ds, batch_size=BATCH_LABELED, shuffle=True, num_workers=NUM_WORKERS,
                      pin_memory=PIN_MEMORY, worker_init_fn=seed_worker, generator=make_generator(SEED))

## The ResNet-18 Encoder (shared by all three methods)

Every method uses the same encoder architecture: torchvision's ResNet-18 built with weights=None, so it starts from random initialization and not from ImageNet. I replace its final fully connected layer with nn.Identity, so the encoder outputs the 512-d vector from the global average pooling layer. Each method then adds its own head on top:

- Part A: a linear layer to 10 classes, trained together with the encoder
- Part B: a linear layer to 4 rotation classes for pretraining, then a separate linear classifier for evaluation
- Part C: an MLP projection head for pretraining, then a separate linear classifier for evaluation

In [ ]:
EMB_DIM = 512


def make_resnet18_encoder():
    encoder = torchvision.models.resnet18(weights=None)  # no pretrained weights
    encoder.fc = nn.Identity()                           # output the 512-d pooled features
    return encoder


def count_params(module):
    return sum(p.numel() for p in module.parameters())


# Checks: correct output size, and the weights really are random rather than pretrained
set_seed(SEED)
enc_a = make_resnet18_encoder()
set_seed(SEED + 1)
enc_b = make_resnet18_encoder()
with torch.no_grad():
    out = enc_a(torch.zeros(2, 3, 96, 96))
same_init = torch.equal(enc_a.conv1.weight, enc_b.conv1.weight)

log("ResNet-18 encoder: weights=None |", f"{count_params(enc_a):,} parameters | output shape {tuple(out.shape)}")
log("Two encoders built with different seeds have identical conv1 weights:", same_init)
assert out.shape == (2, EMB_DIM)
assert not same_init, "weights should come from random initialization, not a fixed pretrained checkpoint"
del enc_a, enc_b

### Training and Evaluation Helpers

These are shared by Part A and both linear evaluations, so all three models get trained and tested the exact same way:

- evaluate runs the model on the full 8,000-image test set and returns the average loss and accuracy.
- train_classifier trains with cross entropy and Adam, and after every epoch records the training loss and accuracy and the test loss and accuracy. Those numbers are what the curves are drawn from.
- LinearProbe wraps a frozen encoder with a new linear layer. Its encoder parameters have requires_grad=False, it runs the encoder under torch.no_grad, and it overrides train() so the encoder always stays in eval mode. Otherwise the BatchNorm running statistics inside the encoder would still get updated, even though no gradients reach its weights.
- encoder_snapshot and check_encoder_unchanged copy the encoder's whole state_dict (weights and BatchNorm buffers) before linear evaluation and compare it afterwards, to prove nothing in the encoder changed.

In [ ]:
criterion = nn.CrossEntropyLoss()


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        total_loss += criterion(logits, y).item() * y.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return total_loss / total, correct / total


def train_classifier(model, params, train_loader, epochs, lr, tag):
    opt = torch.optim.Adam(params, lr=lr, weight_decay=WEIGHT_DECAY)
    history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
    for ep in range(1, epochs + 1):
        model.train()
        t0 = time.time()
        total_loss, correct, total = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            total_loss += loss.item() * y.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            total += y.size(0)
        test_loss, test_acc = evaluate(model, test_loader)
        history["train_loss"].append(total_loss / total)
        history["train_acc"].append(correct / total)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)
        log(f"[{tag} epoch {ep:02d}/{epochs}] train_loss {total_loss / total:.4f} | train_acc {correct / total:.4f} | "
            f"test_loss {test_loss:.4f} | test_acc {test_acc:.4f} | {time.time() - t0:.1f}s")
    return history


class LinearProbe(nn.Module):
    def __init__(self, encoder, num_classes=10):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(EMB_DIM, num_classes)
        for p in self.encoder.parameters():
            p.requires_grad = False

    def train(self, mode=True):
        super().train(mode)
        self.encoder.eval()  # keep BatchNorm statistics frozen too
        return self

    def forward(self, x):
        with torch.no_grad():
            h = self.encoder(x)
        return self.head(h)


def encoder_snapshot(encoder):
    return {k: v.detach().clone() for k, v in encoder.state_dict().items()}


def check_encoder_unchanged(encoder, snapshot, tag):
    current = encoder.state_dict()
    changed = [k for k in snapshot if not torch.equal(snapshot[k], current[k])]
    trainable = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
    log(f"[{tag}] encoder tensors checked: {len(snapshot)} | changed: {len(changed)} | "
        f"encoder params with requires_grad=True: {trainable}")
    assert not changed and trainable == 0, f"encoder changed during linear evaluation: {changed[:5]}"
    print(f"{tag}: the encoder stayed completely frozen during linear evaluation.")


def plot_history(history, title, path):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, history["train_loss"], marker="o", label="train")
    axes[0].plot(epochs, history["test_loss"], marker="o", label="test")
    axes[0].set_title(f"{title}: loss")
    axes[1].plot(epochs, history["train_acc"], marker="o", label="train")
    axes[1].plot(epochs, history["test_acc"], marker="o", label="test")
    axes[1].set_title(f"{title}: accuracy")
    for ax in axes:
        ax.set_xlabel("Epoch")
        ax.grid(True, alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.show()
    log("Saved figure:", path)


run_times = {}

## Part A: Supervised Learning with Limited Labels

This is the baseline. ResNet-18 plus a linear layer to 10 classes, trained end to end from random weights on only the 500 labeled images for 15 epochs. Every parameter in the network gets updated, but 500 images is very little data for an 11M-parameter model, so I expect it to overfit: training accuracy should climb much faster than test accuracy.

In [ ]:
class SupervisedNet(nn.Module):
    def __init__(self, encoder, num_classes=10):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(EMB_DIM, num_classes)

    def forward(self, x):
        return self.head(self.encoder(x))


set_seed(SEED)
model_sup = SupervisedNet(make_resnet18_encoder()).to(device)
labeled_loader = make_labeled_loader()
log(f"Part A: {count_params(model_sup):,} trainable parameters | {len(labeled_loader)} steps per epoch")

t_start = time.time()
hist_sup = train_classifier(model_sup, model_sup.parameters(), labeled_loader, EPOCHS_SUP, LR_SUP, "SUPERVISED")
run_times["partA_supervised"] = time.time() - t_start

sup_test_loss, sup_test_acc = evaluate(model_sup, test_loader)
log(f"Part A supervised test accuracy: {sup_test_acc:.4f} (test loss {sup_test_loss:.4f})")

In [ ]:
plot_history(hist_sup, "Part A supervised (500 labels)", "figures/partA_supervised_curves.png")

torch.save(model_sup.encoder.state_dict(), f"checkpoints/partA_supervised_encoder_seed_{SEED}.pt")
log(f"Saved checkpoints/partA_supervised_encoder_seed_{SEED}.pt")

## Part B: Rotation Self-Supervised Learning

### Rotation dataset

This is the demo's rotation setup. For every unlabeled image, a rotation of 0°, 90°, 180°, or 270° is picked at random, the image is rotated, and the rotation index (0 to 3) becomes the label. No human labels are involved, since the network creates its own task. To predict the rotation, the network has to notice things like where the sky is or which way an animal is standing, and the hope is that those features are also useful for classification.

The transform is the demo's ssl_tf_rot (random crop with padding plus a flip), scaled to 96 pixels. All 100,000 unlabeled images are used.

In [ ]:
ssl_tf_rot = T.Compose([
    T.RandomCrop(96, padding=12),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(mean, std),
])


class RotSSL(Dataset):
    def __init__(self, base, transform):
        self.base = base
        self.transform = transform

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, _ = self.base[idx]             # ignore the (-1) label
        rot_label = random.randint(0, 3)    # 0..3
        img_rot = T.functional.rotate(img, 90 * rot_label)
        x = self.transform(img_rot)
        return x, torch.tensor(rot_label, dtype=torch.long)


set_seed(SEED)
rot_ds = RotSSL(unlabeled_raw, ssl_tf_rot)
rot_loader = DataLoader(rot_ds, batch_size=BATCH_ROT, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                        worker_init_fn=seed_worker, generator=make_generator(SEED))
log(f"Rotation dataset: {len(rot_ds)} unlabeled images | {len(rot_loader)} steps per epoch")

Like in the demo, here is the same image at all four rotations, followed by a check on one batch: the shapes, and that every rotation label is between 0 and 3.

In [ ]:
img0, _ = unlabeled_raw[0]
rot_degs = [0, 90, 180, 270]
show_pil_row([T.functional.rotate(img0, d) for d in rot_degs], [f"{d}°" for d in rot_degs], figsize=(10, 3))

x, r = next(iter(rot_loader))
print("Rotation batch x:", tuple(x.shape), "Rotation labels r:", tuple(r.shape))
print("First 16 rotation labels:", r[:16].tolist())
assert r.min().item() >= 0 and r.max().item() <= 3
print("Rotation labels are in the range 0-3.")

### Rotation model = encoder + rotation head

The rotation head is one linear layer from the 512-d embedding to 4 rotation classes. One forward pass to check the shapes, same as the demo.

In [ ]:
set_seed(SEED)
encoder_rot = make_resnet18_encoder().to(device)
rot_head = nn.Linear(EMB_DIM, 4).to(device)

x, r = x.to(device), r.to(device)
with torch.no_grad():
    z = encoder_rot(x)
    logits = rot_head(z)
print("x:", tuple(x.shape))
print("z (embedding):", tuple(z.shape))
print("logits:", tuple(logits.shape))

### Train rotation SSL

The training loop is the demo's, run for 15 epochs. It also counts how many times each rotation label shows up across the whole run. That is an easy check that the labels really are 0 to 3 and that all four rotations appear about equally often.

In [ ]:
opt = torch.optim.Adam(list(encoder_rot.parameters()) + list(rot_head.parameters()), lr=LR_ROT, weight_decay=WEIGHT_DECAY)
crit = nn.CrossEntropyLoss()
hist_rot = {"loss": [], "acc": []}
rot_label_counts = torch.zeros(4, dtype=torch.long)

t_start = time.time()
for ep in range(1, EPOCHS_ROT + 1):
    encoder_rot.train(); rot_head.train()
    losses, accs = [], []
    t0 = time.time()

    for step, (x, r) in enumerate(rot_loader):
        rot_label_counts += torch.bincount(r, minlength=4)
        x, r = x.to(device), r.to(device)
        z = encoder_rot(x)
        logits = rot_head(z)
        loss = crit(logits, r)

        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

        losses.append(loss.item())
        accs.append((logits.argmax(1) == r).float().mean().item())

        if step % 100 == 0:
            print(f"  step {step:03d} | loss {loss.item():.3f} | batch_acc {accs[-1]:.3f}")

    hist_rot["loss"].append(float(np.mean(losses)))
    hist_rot["acc"].append(float(np.mean(accs)))
    log(f"[ROT SSL epoch {ep:02d}/{EPOCHS_ROT}] mean_loss {np.mean(losses):.4f} | mean_acc {np.mean(accs):.4f} | {time.time() - t0:.1f}s")

run_times["partB_rotation_pretraining"] = time.time() - t_start
log("Rotation label counts over all of training (0/90/180/270):", rot_label_counts.tolist())
assert rot_label_counts.numel() == 4
print("Rotation SSL training done.")

In [ ]:
epochs = range(1, EPOCHS_ROT + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, hist_rot["loss"], marker="o")
axes[0].set_title("Part B rotation pretraining: loss")
axes[1].plot(epochs, hist_rot["acc"], marker="o")
axes[1].set_title("Part B rotation pretraining: rotation accuracy")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("figures/partB_rotation_pretraining.png", dpi=150)
plt.show()
log("Saved figure: figures/partB_rotation_pretraining.png")

torch.save(encoder_rot.state_dict(), f"checkpoints/partB_rotation_encoder_seed_{SEED}.pt")
log(f"Saved checkpoints/partB_rotation_encoder_seed_{SEED}.pt")

### Linear evaluation on the frozen rotation encoder

Now the rotation head is thrown away, since it only knows about rotations, not classes. The encoder gets frozen completely, and only a new linear layer (512 to 10) is trained on the same 500 labeled images for 20 epochs. The optimizer only receives the linear layer's parameters. I take a snapshot of the encoder first and compare against it afterwards, to confirm that none of its weights or BatchNorm statistics changed.

In [ ]:
del rot_head  # the rotation head is not used anymore

set_seed(SEED)
probe_rot = LinearProbe(encoder_rot).to(device)
snapshot_rot = encoder_snapshot(probe_rot.encoder)
log(f"Part B linear probe: trainable parameters {sum(p.numel() for p in probe_rot.parameters() if p.requires_grad):,} "
    f"(head only) | frozen encoder parameters {count_params(probe_rot.encoder):,}")

labeled_loader = make_labeled_loader()
t_start = time.time()
hist_rot_lin = train_classifier(probe_rot, probe_rot.head.parameters(), labeled_loader, EPOCHS_LIN, LR_LIN, "ROT LINEAR")
run_times["partB_linear_eval"] = time.time() - t_start

check_encoder_unchanged(probe_rot.encoder, snapshot_rot, "Part B")
rot_test_loss, rot_test_acc = evaluate(probe_rot, test_loader)
log(f"Part B rotation SSL linear-evaluation test accuracy: {rot_test_acc:.4f} (test loss {rot_test_loss:.4f})")

In [ ]:
plot_history(hist_rot_lin, "Part B rotation SSL linear evaluation", "figures/partB_linear_eval_curves.png")

## Part C: SimCLR-Style Contrastive Learning

### The 20,000-image subset and the two views

To keep SimCLR manageable on a Colab GPU, I use 20,000 of the 100,000 unlabeled images, picked without replacement by a NumPy generator seeded with SEED.

Each image gets augmented twice with the demo's four augmentations: RandomResizedCrop (scale 0.6 to 1.0), RandomHorizontalFlip (p = 0.5), ColorJitter (0.4, 0.4, 0.4, 0.1), and RandomGrayscale (p = 0.2). I kept the demo's settings and only changed the crop size from 32 to 96. The two views of the same image are a positive pair, and the views of every other image in the batch are negatives.

In [ ]:
aug = T.Compose([
    T.RandomResizedCrop(96, scale=(0.6, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(0.4, 0.4, 0.4, 0.1),
    T.RandomGrayscale(p=0.2),
    T.ToTensor(),
    T.Normalize(mean, std),
])


class TwoViews(Dataset):
    def __init__(self, base, aug):
        self.base = base
        self.aug = aug

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, _ = self.base[idx]
        x1 = self.aug(img)
        x2 = self.aug(img)
        return x1, x2


SIMCLR_IDX = sorted(np.random.default_rng(SEED).choice(len(unlabeled_raw), size=SIMCLR_SUBSET_SIZE, replace=False).tolist())
assert len(SIMCLR_IDX) == len(set(SIMCLR_IDX)) == SIMCLR_SUBSET_SIZE
log(f"SimCLR subset: {len(SIMCLR_IDX)} unlabeled images | first 10 indices {SIMCLR_IDX[:10]}")

set_seed(SEED)
simclr_ds = TwoViews(Subset(unlabeled_raw, SIMCLR_IDX), aug)
simclr_loader = DataLoader(simclr_ds, batch_size=BATCH_SIM, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                           drop_last=True, worker_init_fn=seed_worker, generator=make_generator(SEED))
log(f"SimCLR dataset: {len(simclr_ds)} images | batch {BATCH_SIM} | {len(simclr_loader)} steps per epoch")

In [ ]:
img0, _ = simclr_ds.base[0]
v1 = aug(img0)
v2 = aug(img0)
show_pil_row([img0], ["Original"], figsize=(3, 3))
show_tensor_row([v1, v2], ["Aug view 1", "Aug view 2"], figsize=(6, 3))

x1, x2 = next(iter(simclr_loader))
print("SimCLR batch x1:", tuple(x1.shape), "x2:", tuple(x2.shape))

### SimCLR model = encoder + MLP projection head

Same as the demo: the encoder produces the representation, and a small MLP projection head (512 → 512 → ReLU → 128) maps it into the space where the contrastive loss is computed. The projection head is only used during pretraining. Afterwards I throw it away and keep the encoder's 512-d output, which is the standard SimCLR approach.

In [ ]:
set_seed(SEED)
encoder_sim = make_resnet18_encoder().to(device)
proj = nn.Sequential(
    nn.Linear(EMB_DIM, 512), nn.ReLU(inplace=True),
    nn.Linear(512, 128),
).to(device)

x1, x2 = x1.to(device), x2.to(device)
with torch.no_grad():
    h1, h2 = encoder_sim(x1), encoder_sim(x2)
    z1, z2 = proj(h1), proj(h2)
print("h1/h2 (encoder embeddings):", tuple(h1.shape), tuple(h2.shape))
print("z1/z2 (projected):", tuple(z1.shape), tuple(z2.shape))

### NT-Xent (InfoNCE) loss with cosine similarity and tau = 0.2

This is the demo's nt_xent function, unchanged. Both projected views are L2-normalized, so the dot product z·zᵀ is exactly the cosine similarity between every pair of the 2B samples, and dividing by tau = 0.2 sharpens it. Each sample's own similarity is masked out, its positive is the other view of the same image (B positions away), and the loss is cross entropy of picking that positive out of all the other samples.

The check below compares the matrix the loss uses against F.cosine_similarity computed directly, to confirm that it really is cosine similarity.

In [ ]:
def nt_xent(z1, z2, tau=0.2):
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    B = z1.size(0)

    z = torch.cat([z1, z2], dim=0)         # [2B,D]
    sim = (z @ z.T) / tau                  # [2B,2B]

    # mask diagonal (self similarity)
    eye = torch.eye(2 * B, device=z.device, dtype=torch.bool)
    sim = sim.masked_fill(eye, -1e9)

    # positives at offsets ±B
    pos = torch.cat([torch.diag(sim, B), torch.diag(sim, -B)], dim=0)  # [2B]
    denom = torch.logsumexp(sim, dim=1)                                # [2B]
    return (-pos + denom).mean()


with torch.no_grad():
    Z = torch.cat([F.normalize(z1, dim=1), F.normalize(z2, dim=1)], dim=0)
    sim_used = Z @ Z.T
    zz = torch.cat([z1, z2], dim=0)
    sim_direct = F.cosine_similarity(zz.unsqueeze(1), zz.unsqueeze(0), dim=-1)
    max_diff = (sim_used - sim_direct).abs().max().item()
    print("Similarity matrix shape:", tuple(sim_used.shape))
    print("Top-left 4x4 of the cosine similarities / tau:")
    print((sim_used[:4, :4] / TAU).cpu().numpy().round(3))
    print("Max difference from F.cosine_similarity:", max_diff)
    print("NT-Xent loss on one batch before training:", float(nt_xent(z1, z2, tau=TAU)))
assert max_diff < 1e-4

### Train SimCLR

The demo's training loop, run for 20 epochs on the 20,000-image subset.

In [ ]:
opt = torch.optim.Adam(list(encoder_sim.parameters()) + list(proj.parameters()), lr=LR_SIM, weight_decay=WEIGHT_DECAY)
hist_sim = {"loss": []}

t_start = time.time()
for ep in range(1, EPOCHS_SIM + 1):
    encoder_sim.train(); proj.train()
    losses = []
    t0 = time.time()

    for step, (x1, x2) in enumerate(simclr_loader):
        x1, x2 = x1.to(device), x2.to(device)
        z1 = proj(encoder_sim(x1))
        z2 = proj(encoder_sim(x2))
        loss = nt_xent(z1, z2, tau=TAU)

        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

        losses.append(loss.item())
        if step % 25 == 0:
            print(f"  step {step:03d} | loss {loss.item():.3f}")

    hist_sim["loss"].append(float(np.mean(losses)))
    log(f"[SimCLR epoch {ep:02d}/{EPOCHS_SIM}] mean_loss {np.mean(losses):.4f} | {time.time() - t0:.1f}s")

run_times["partC_simclr_pretraining"] = time.time() - t_start
print("SimCLR training done.")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, EPOCHS_SIM + 1), hist_sim["loss"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Mean NT-Xent loss")
plt.title("Part C SimCLR pretraining loss (tau = 0.2)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("figures/partC_simclr_pretraining_loss.png", dpi=150)
plt.show()
log("Saved figure: figures/partC_simclr_pretraining_loss.png")

torch.save(encoder_sim.state_dict(), f"checkpoints/partC_simclr_encoder_seed_{SEED}.pt")
log(f"Saved checkpoints/partC_simclr_encoder_seed_{SEED}.pt")

### Linear evaluation on the frozen SimCLR encoder

Same procedure as Part B: the projection head is removed, the encoder is frozen, and only a new linear layer is trained on the same 500 labeled images for 20 epochs, followed by the same check that the encoder did not change.

In [ ]:
del proj  # the projection head is not used anymore

set_seed(SEED)
probe_sim = LinearProbe(encoder_sim).to(device)
snapshot_sim = encoder_snapshot(probe_sim.encoder)
log(f"Part C linear probe: trainable parameters {sum(p.numel() for p in probe_sim.parameters() if p.requires_grad):,} "
    f"(head only) | frozen encoder parameters {count_params(probe_sim.encoder):,}")

labeled_loader = make_labeled_loader()
t_start = time.time()
hist_sim_lin = train_classifier(probe_sim, probe_sim.head.parameters(), labeled_loader, EPOCHS_LIN, LR_LIN, "SimCLR LINEAR")
run_times["partC_linear_eval"] = time.time() - t_start

check_encoder_unchanged(probe_sim.encoder, snapshot_sim, "Part C")
sim_test_loss, sim_test_acc = evaluate(probe_sim, test_loader)
log(f"Part C SimCLR linear-evaluation test accuracy: {sim_test_acc:.4f} (test loss {sim_test_loss:.4f})")

In [ ]:
plot_history(hist_sim_lin, "Part C SimCLR linear evaluation", "figures/partC_linear_eval_curves.png")

## Part D: Nearest-Neighbor Visualization

### Test-set embeddings for all three encoders

This is the demo's compute_embeddings function. It runs all 8,000 test images through each encoder (the heads are not used), and L2-normalizes the 512-d embeddings. After normalization, a dot product between two embeddings is their cosine similarity. For the supervised model I use only its encoder, the part before the final linear layer, so all three are compared with the same 512-d features.

In [ ]:
@torch.no_grad()
def compute_embeddings(encoder, loader):
    encoder.eval()
    embs = []
    for x, _ in loader:
        x = x.to(device)
        z = encoder(x)                  # [B,512]
        z = F.normalize(z, dim=1)
        embs.append(z.cpu())
    return torch.cat(embs, dim=0)       # [N,512]


encoders = {
    "Supervised": model_sup.encoder,
    "Rotation SSL": encoder_rot,
    "SimCLR": encoder_sim,
}
embeddings = {name: compute_embeddings(enc, test_loader) for name, enc in encoders.items()}
test_labels = np.array(test_raw.labels)

for name, E in embeddings.items():
    norms = E.norm(dim=1)
    log(f"{name} test embeddings: {tuple(E.shape)} | norm min/max {norms.min():.4f}/{norms.max():.4f}")
    assert E.shape == (len(test_raw), EMB_DIM) and torch.allclose(norms, torch.ones_like(norms), atol=1e-4)

### Picking the query images

I want queries from different classes, picked by a fixed rule instead of by hand:

1. the first test image at index SLICE (486) or later whose class is CLS_A (6 = horse)
2. the first test image at index SLICE or later whose class is CLS_B (0 = airplane)
3. the first test image at index SLICE or later whose class is neither of those

The same QUERY_INDICES list is used for all three encoders.

In [ ]:
def first_index_from(start, condition):
    return next(i for i in range(start, len(test_labels)) if condition(test_labels[i]))


QUERY_INDICES = [
    first_index_from(SLICE, lambda y: y == CLS_A),
    first_index_from(SLICE, lambda y: y == CLS_B),
    first_index_from(SLICE, lambda y: y not in (CLS_A, CLS_B)),
]
assert len(set(test_labels[QUERY_INDICES].tolist())) == 3
log("Query indices:", QUERY_INDICES, "| classes:", [classes[test_labels[i]] for i in QUERY_INDICES])

### Top-5 neighbors by cosine similarity

For each query and each encoder, the similarity to every test image is E @ q (cosine similarity, since everything is normalized). I take the top 6 and drop the query itself, which leaves the 5 nearest neighbors. Like the demo, it prints and shows the neighbor's class and cosine score.

To make the three encoders easy to compare, each query gets one figure with three rows (Supervised, Rotation SSL, SimCLR). The first column is the query and the next five are its neighbors. A neighbor's title is green if it has the same class as the query and red if not. I also count how many of the 5 neighbors share the query's class, which makes the figures easier to talk about.

In [ ]:
def top5_neighbors(E, qi):
    q_emb = E[qi]
    sims = E @ q_emb                                    # cosine similarity to every test image
    top = torch.topk(sims, k=6).indices.tolist()
    top = [j for j in top if j != qi][:5]               # drop the query itself
    # double-check against F.cosine_similarity for the best neighbor
    direct = F.cosine_similarity(E[qi].unsqueeze(0), E[top[0]].unsqueeze(0)).item()
    assert abs(direct - sims[top[0]].item()) < 1e-4
    return top, sims


nn_results = {}
for k, qi in enumerate(QUERY_INDICES, start=1):
    q_img, q_y = test_raw[qi]
    fig, axes = plt.subplots(len(encoders), 6, figsize=(15, 10.5))
    nn_results[qi] = {}
    for row, name in enumerate(encoders):
        top, sims = top5_neighbors(embeddings[name], qi)
        same = sum(int(test_labels[j] == q_y) for j in top)
        nn_results[qi][name] = {
            "neighbors": top,
            "neighbor_classes": [classes[test_labels[j]] for j in top],
            "cosine": [round(float(sims[j]), 4) for j in top],
            "same_class_count": same,
        }
        log(f"query {qi} ({classes[q_y]}) | {name:<12} | neighbors {top} | "
            f"classes {[classes[test_labels[j]] for j in top]} | cos {[round(float(sims[j]), 3) for j in top]} | same class {same}/5")

        axes[row, 0].imshow(q_img, interpolation="nearest")
        axes[row, 0].set_title(f"QUERY idx={qi}\n{classes[q_y]}", fontsize=9)
        axes[row, 0].set_ylabel(name, fontsize=11)
        for col, j in enumerate(top, start=1):
            img_j, y_j = test_raw[j]
            axes[row, col].imshow(img_j, interpolation="nearest")
            axes[row, col].set_title(f"idx={j}\n{classes[y_j]}\ncos={float(sims[j]):.2f}", fontsize=9,
                                     color="green" if y_j == q_y else "red")
        for ax in axes[row]:
            ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f"Query {k}: top-5 cosine nearest neighbors (rows: Supervised, Rotation SSL, SimCLR)", fontsize=12)
    plt.tight_layout()
    path = f"figures/partD_nearest_neighbors_query{k}.png"
    plt.savefig(path, dpi=130)
    plt.show()
    log("Saved figure:", path)

## Results Summary

All of the main numbers in one table, and also saved to results/hw6_metrics.json together with the training histories, settings, and run times.

In [ ]:
summary_rows = [
    ("A: Supervised (end to end)", EPOCHS_SUP, "-", sup_test_acc),
    ("B: Rotation SSL + linear", EPOCHS_LIN, EPOCHS_ROT, rot_test_acc),
    ("C: SimCLR + linear", EPOCHS_LIN, EPOCHS_SIM, sim_test_acc),
]
print(f"{'Model':<30}{'Labeled epochs':>16}{'Pretrain epochs':>17}{'Test accuracy':>15}")
for name, lab_ep, pre_ep, acc in summary_rows:
    print(f"{name:<30}{lab_ep:>16}{str(pre_ep):>17}{acc:>15.4f}")

same_class_table = {name: [nn_results[qi][name]["same_class_count"] for qi in QUERY_INDICES] for name in encoders}
print("\nSame-class neighbors out of 5 for each query", [classes[test_labels[i]] for i in QUERY_INDICES])
for name, counts in same_class_table.items():
    print(f"  {name:<12} {counts}")

metrics = {
    "personal_parameters": {"SID4": SID4, "SEED": SEED, "SLICE": SLICE, "HP_ID": HP_ID, "CLS_A": CLS_A, "CLS_B": CLS_B},
    "device": torch.cuda.get_device_name(0) if device == "cuda" else "cpu",
    "versions": {"torch": torch.__version__, "torchvision": torchvision.__version__},
    "data": {"labeled_subset_size": len(LABELED_IDX), "labeled_subset_fingerprint": LABELED_HASH,
             "labeled_indices": LABELED_IDX, "rotation_unlabeled_size": len(rot_ds),
             "simclr_subset_size": len(SIMCLR_IDX), "test_size": len(test_raw), "image_size": 96,
             "normalization_mean": mean, "normalization_std": std},
    "settings": {"encoder": "torchvision resnet18, weights=None, fc=Identity, 512-d",
                 "BATCH_LABELED": BATCH_LABELED, "EPOCHS_SUP": EPOCHS_SUP, "LR_SUP": LR_SUP,
                 "BATCH_ROT": BATCH_ROT, "EPOCHS_ROT": EPOCHS_ROT, "LR_ROT": LR_ROT,
                 "BATCH_SIM": BATCH_SIM, "EPOCHS_SIM": EPOCHS_SIM, "LR_SIM": LR_SIM, "TAU": TAU,
                 "EPOCHS_LIN": EPOCHS_LIN, "LR_LIN": LR_LIN, "WEIGHT_DECAY": WEIGHT_DECAY, "optimizer": "Adam"},
    "test_accuracy": {"supervised": sup_test_acc, "rotation_linear_eval": rot_test_acc, "simclr_linear_eval": sim_test_acc},
    "test_loss": {"supervised": sup_test_loss, "rotation_linear_eval": rot_test_loss, "simclr_linear_eval": sim_test_loss},
    "histories": {"supervised": hist_sup, "rotation_pretraining": hist_rot, "rotation_linear_eval": hist_rot_lin,
                  "simclr_pretraining": hist_sim, "simclr_linear_eval": hist_sim_lin},
    "rotation_label_counts": rot_label_counts.tolist(),
    "query_indices": QUERY_INDICES,
    "nearest_neighbors": {str(qi): v for qi, v in nn_results.items()},
    "run_times_seconds": {k: round(v, 1) for k, v in run_times.items()},
}
with open("results/hw6_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
log("Saved results/hw6_metrics.json | run times (s):", metrics["run_times_seconds"])

### Packaging the outputs

The figures, the metrics file, and the raw log are written to the runtime's local disk, which Colab does not sync back to GitHub. This cell zips them up and, on Colab, starts a download, so they can be unzipped into the HW6 folder of the repo. The checkpoints are left out because .pt files are ignored by the repo's .gitignore.

In [ ]:
import shutil

os.makedirs("hw6_outputs", exist_ok=True)
for folder in ["figures", "results"]:
    shutil.copytree(folder, os.path.join("hw6_outputs", folder), dirs_exist_ok=True)
archive = shutil.make_archive("hw6_outputs", "zip", "hw6_outputs")
log("Packaged outputs:", archive, "|", sorted(os.listdir("hw6_outputs/figures")))

try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Not running on Colab, so the zip is just left at", archive)

## Final Comparison

FINAL_COMPARISON_PLACEHOLDER: written after the full Colab run, based only on the outputs above.